In [74]:
from datetime import timedelta
from pathlib import Path

import numpy as np
import polars as pl
import torch
from torch import nn


root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

raw_dir = root / "data" / "raw"
feature_groups = ("transactions",)
transaction_window = 56
run_window_comparison = False
available_groups = {"transactions", "calendar", "holidays", "oil", "store", "last_year"}
if unknown_groups := set(feature_groups) - available_groups:
    raise ValueError(f"Unknown feature groups: {', '.join(sorted(unknown_groups))}")
if transaction_window < 1:
    raise ValueError("transaction_window must be positive.")

output_dir = root / "data" / "output" / ("-".join(feature_groups) or "baseline")

lookback = 56
origin_stride = 5
seed = 42
epochs = 8
batch_size = 1024
learning_rate = 0.001
hidden_widths = (128, 64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
print(f"Training device: {device}", flush=True)

Training device: cuda


## Join the competition data

In [75]:
train = pl.read_csv(raw_dir / "train.csv", try_parse_dates=True)
test = pl.read_csv(raw_dir / "test.csv", try_parse_dates=True)
stores = pl.read_csv(raw_dir / "stores.csv").sort("store_nbr")
oil = pl.read_csv(raw_dir / "oil.csv", try_parse_dates=True)
transactions = pl.read_csv(raw_dir / "transactions.csv", try_parse_dates=True)
events = pl.read_csv(raw_dir / "holidays_events.csv", try_parse_dates=True)
sample = pl.read_csv(raw_dir / "sample_submission.csv")

series_columns = ["store_nbr", "family"]
series_day = series_columns + ["date"]
store_day = ["store_nbr", "date"]
series_keys = train.select(series_columns).unique().sort(series_columns)
first_date = train["date"].min()
last_labeled_date = train["date"].max()
test_start = last_labeled_date + timedelta(days=1)
test_dates = test["date"].unique().sort()
horizon = len(test_dates)
validation_start = test_start - timedelta(days=horizon)
split_day = (validation_start - first_date).days
test_day = (test_start - first_date).days

assert test_dates.to_list() == pl.date_range(test_start, test_dates[-1], eager=True).to_list()
assert test.height == series_keys.height * horizon
assert test.select(series_columns).unique().sort(series_columns).equals(series_keys)
assert set(series_keys["store_nbr"]).issubset(stores["store_nbr"])

calendar = pl.DataFrame({"date": pl.date_range(first_date, test_dates[-1], eager=True)})
sales_rows = pl.concat([
    train.with_columns(pl.lit("train").alias("split")),
    test.with_columns(pl.lit(None, dtype=train.schema["sales"]).alias("sales"))
    .select(train.columns)
    .with_columns(pl.lit("test").alias("split")),
])
assert sales_rows["id"].n_unique() == sales_rows.height
assert sales_rows.select(series_day).n_unique() == sales_rows.height

holiday_types = ("Holiday", "Transfer", "Additional", "Bridge")
scheduled = events.filter(
    ~pl.col("transferred") & pl.col("type").is_in((*holiday_types, "Work Day"))
)
holiday_rows = [
    scheduled.filter(pl.col("locale") == "National").join(
        stores.select("store_nbr"), how="cross"
    ).select(store_day + ["type"])
]
for scope, location in [("Regional", "state"), ("Local", "city")]:
    holiday_rows.append(
        scheduled.filter(pl.col("locale") == scope)
        .join(stores.select("store_nbr", pl.col(location).alias("locale_name")), on="locale_name")
        .select(store_day + ["type"])
    )
holiday_flags = pl.concat(holiday_rows).group_by(store_day).agg(
    pl.col("type").is_in(holiday_types).any().alias("holiday"),
    (pl.col("type") == "Work Day").any().alias("work_day"),
)

store_calendar = (
    stores.join(calendar, how="cross")
    .join(oil, on="date", how="left", validate="m:1")
    .join(transactions, on=store_day, how="left", validate="1:1")
    .join(holiday_flags, on=store_day, how="left", validate="1:1")
    .with_columns(pl.col("holiday", "work_day").fill_null(False))
    .sort(store_day)
)
unified = sales_rows.join(store_calendar, on=store_day, how="left", validate="m:1")
assert unified.height == sales_rows.height
assert unified.filter(pl.col("split") == "test")["sales"].null_count() == test.height

print(f"Unified table: {unified.height:,} rows, {unified.width} columns")
print(f"Training targets through {validation_start - timedelta(days=1)}")
print(f"Validation: {validation_start} to {last_labeled_date}")
print(f"Test: {test_dates[0]} to {test_dates[-1]}")
unified.head()

Unified table: 3,029,400 rows, 15 columns
Training targets through 2017-07-30
Validation: 2017-07-31 to 2017-08-15
Test: 2017-08-16 to 2017-08-31


id,date,store_nbr,family,sales,onpromotion,split,city,state,type,cluster,dcoilwtico,transactions,holiday,work_day
i64,date,i64,str,f64,i64,str,str,str,str,i64,f64,i64,bool,bool
0,2013-01-01,1,"""AUTOMOTIVE""",0.0,0,"""train""","""Quito""","""Pichincha""","""D""",13,null,null,true,false
1,2013-01-01,1,"""BABY CARE""",0.0,0,"""train""","""Quito""","""Pichincha""","""D""",13,null,null,true,false
2,2013-01-01,1,"""BEAUTY""",0.0,0,"""train""","""Quito""","""Pichincha""","""D""",13,null,null,true,false
3,2013-01-01,1,"""BEVERAGES""",0.0,0,"""train""","""Quito""","""Pichincha""","""D""",13,null,null,true,false
4,2013-01-01,1,"""BOOKS""",0.0,0,"""train""","""Quito""","""Pichincha""","""D""",13,null,null,true,false


In [76]:
series_count = series_keys.height
day_count = calendar.height
store_count = stores.height
series_store = series_keys.join(
    stores.select("store_nbr").with_row_index("store_idx"),
    on="store_nbr", how="left", validate="m:1", maintain_order="left",
)["store_idx"]

panel = (
    series_keys.join(calendar, how="cross")
    .join(unified.select(series_day + ["sales", "onpromotion"]),
          on=series_day, how="left", validate="1:1")
    .sort(series_day)
)

oil_context = (
    calendar.join(oil, on="date", how="left", validate="1:1")
    .sort("date")
    .with_columns(pl.col("dcoilwtico").forward_fill().alias("observed_oil"))
    .select(
        "date",
        pl.col("observed_oil").shift(1).alias("last_oil_price"),
        (pl.col("observed_oil").shift(1) - pl.col("observed_oil").shift(8))
        .alias("oil_change_7d"),
    )
)

store_features = (
    store_calendar.with_columns(
        (pl.col("date").dt.day() == 15).alias("midmonth_payday"),
        (pl.col("date") == pl.col("date").dt.month_end()).alias("month_end_payday"),
    )
    .with_columns([
        (pl.col("date").dt.weekday() == day).alias(f"weekday_{day}")
        for day in range(1, 8)
    ])
)

calendar_names = []
if "calendar" in feature_groups:
    calendar_names.extend([f"weekday_{day}" for day in range(1, 8)])
    calendar_names.extend(["midmonth_payday", "month_end_payday"])
if "holidays" in feature_groups:
    calendar_names.extend(["holiday", "work_day"])

oil_names = ["last_oil_price", "oil_change_7d"]
transaction_names = ["log_transactions_mean", "transaction_record_coverage"]
context_names = (oil_names if "oil" in feature_groups else []) + (
    transaction_names if "transactions" in feature_groups else []
)
numeric_width = len(context_names) - int("transactions" in feature_groups)
store_categories = stores.select("type", "cluster").to_dummies()
store_feature_names = store_categories.columns if "store" in feature_groups else []


def tensor(values, dtype=None):
    return torch.as_tensor(np.array(values, copy=True), dtype=dtype, device=device)


panel_shape = (series_count, day_count)
log_sales = tensor(panel["sales"].fill_null(0).log1p().to_numpy().reshape(panel_shape), torch.float32)
sales_observed = tensor(panel["sales"].is_not_null().to_numpy().reshape(panel_shape))
log_promotions = tensor(
    panel["onpromotion"].fill_null(0).log1p().to_numpy().reshape(panel_shape), torch.float32
)
series_store_idx = tensor(series_store.to_numpy(), torch.long)

future_calendar = torch.empty((store_count, day_count, 0), device=device)
if calendar_names:
    future_calendar = tensor(
        store_features.select(calendar_names).to_numpy().reshape(store_count, day_count, -1),
        torch.float32,
    )

static_features = torch.empty((store_count, 0), device=device)
if store_feature_names:
    static_features = tensor(store_categories.to_numpy(), torch.float32)


def make_raw_context(window):
    components = []
    if "oil" in feature_groups:
        daily_oil = oil_context.select(oil_names).cast(pl.Float32).to_numpy()
        components.append(np.broadcast_to(daily_oil, (store_count, day_count, len(oil_names))))
    if "transactions" in feature_groups:
        transactions_by_origin = store_calendar.select(
            pl.col("transactions").rolling_mean(window, min_samples=1).shift(1)
            .over("store_nbr").log1p().alias(transaction_names[0]),
            (
                pl.col("transactions").is_not_null().cast(pl.Float32)
                .rolling_sum(window, min_samples=1).shift(1).over("store_nbr") / window
            ).fill_null(0).alias(transaction_names[1]),
        ).cast(pl.Float32)
        components.append(transactions_by_origin.to_numpy().reshape(store_count, day_count, -1))

    if components:
        return np.concatenate(components, axis=-1)
    return np.empty((store_count, day_count, 0), dtype=np.float32)


raw_context = make_raw_context(transaction_window)
assert sales_observed[:, split_day:test_day].all()
assert not sales_observed[:, test_day:].any()
print(f"Calendar panel: {series_count:,} series × {day_count:,} days")

Calendar panel: 1,782 series × 1,704 days


In [77]:
history_offsets = torch.arange(-lookback, 0, device=device)
target_offsets = torch.arange(horizon, device=device)
annual_lag = 364
embedding_size = 16
base_input_width = (
    2 * lookback + embedding_size + horizon * (1 + len(calendar_names))
    + len(context_names) + len(store_feature_names)
)


def training_origins(target_cutoff, include_last=False):
    last_origin = target_cutoff - horizon
    origins = np.arange(lookback, last_origin + 1, origin_stride)
    if include_last and origins[-1] != last_origin:
        origins = np.append(origins, last_origin)
    return origins


def fit_context(origins, raw_features):
    if not numeric_width:
        return tensor(raw_features), {}

    numeric = raw_features[..., :numeric_width]
    training_values = numeric[:, origins, :]
    mean = np.nanmean(training_values, axis=(0, 1))
    std = np.nanstd(training_values, axis=(0, 1))
    std = np.where(std > 0, std, 1)
    scaled = np.nan_to_num((numeric - mean) / std, nan=0.0)
    values = np.concatenate([scaled, raw_features[..., numeric_width:]], axis=-1)
    return tensor(values), {"mean": mean.tolist(), "std": std.tolist()}


def model_inputs(origins, series_ids, context, include_last_year):
    rows = series_ids[:, None]
    past = origins[:, None] + history_offsets
    future = origins[:, None] + target_offsets
    store_ids = series_store_idx[series_ids]
    history_context = context[store_ids, origins]
    if include_last_year:
        previous_year = future - annual_lag
        lookup_days = previous_year.clamp_min(0)
        observed_last_year = (previous_year >= 0) & sales_observed[rows, lookup_days]
        sales_last_year = log_sales[rows, lookup_days] * observed_last_year
        history_context = torch.cat([
            history_context, sales_last_year, observed_last_year.float(),
        ], dim=1)
    return (
        series_ids,
        log_sales[rows, past],
        sales_observed[rows, past],
        log_promotions[rows, future],
        future_calendar[store_ids[:, None], future].flatten(1),
        history_context,
        static_features[store_ids],
    )


class SalesForecaster(nn.Module):
    def __init__(self, include_last_year):
        super().__init__()
        self.include_last_year = include_last_year
        self.input_width = base_input_width + 2 * horizon * int(include_last_year)
        self.series_embedding = nn.Embedding(series_count, embedding_size)
        first, second = hidden_widths
        self.network = nn.Sequential(
            nn.Linear(self.input_width, first),
            nn.ReLU(),
            nn.Linear(first, second),
            nn.ReLU(),
            nn.Linear(second, horizon),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)
        self.register_buffer("weekday_index", torch.arange(horizon) % 7)

    def forward(self, series_id, past_sales, past_observed, future_promotions,
                calendar_features, historical_context, store_metadata):
        recent_sales = past_sales[:, -28:].reshape(-1, 4, 7)
        recent_observed = past_observed[:, -28:].reshape(-1, 4, 7)
        weekday_mean = (
            (recent_sales * recent_observed).sum(dim=1)
            / recent_observed.sum(dim=1).clamp_min(1)
        )
        baseline = weekday_mean[:, self.weekday_index]
        features = torch.cat([
            past_sales, past_observed.float(), future_promotions,
            self.series_embedding(series_id), calendar_features,
            historical_context, store_metadata,
        ], dim=1)
        return baseline + self.network(features)


@torch.inference_mode()
def predict(model, origin, context):
    model.eval()
    series_ids = torch.arange(series_count, device=device)
    origins = torch.full_like(series_ids, origin)
    return model(*model_inputs(origins, series_ids, context, model.include_last_year)).clamp_min(0)


def forecast_rmsle(predictions, origin):
    dates = slice(origin, origin + horizon)
    mask = sales_observed[:, dates]
    errors = predictions[mask] - log_sales[:, dates][mask]
    return errors.square().mean().sqrt().item()


def fit_model(origins, context, epoch_count, validation_origin=None,
              *, model_seed=seed, include_last_year=None, verbose=True):
    if include_last_year is None:
        include_last_year = "last_year" in feature_groups
    torch.manual_seed(model_seed)
    rng = np.random.default_rng(model_seed)
    model = SalesForecaster(include_last_year).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    example_count = len(origins) * series_count
    origins_on_device = tensor(origins, torch.long)
    best_score, best_epoch, best_weights = float("inf"), 0, None

    for epoch in range(1, epoch_count + 1):
        model.train()
        shuffled = tensor(rng.permutation(example_count), torch.long)
        epoch_error = torch.zeros((), device=device)
        epoch_observed = torch.zeros((), device=device)

        for ids in shuffled.split(batch_size):
            series_ids = ids % series_count
            batch_origins = origins_on_device[ids // series_count]
            future = batch_origins[:, None] + target_offsets
            target = log_sales[series_ids[:, None], future]
            mask = sales_observed[series_ids[:, None], future]

            optimizer.zero_grad(set_to_none=True)
            predictions = model(*model_inputs(batch_origins, series_ids, context, include_last_year))
            error_sum = ((predictions - target).square() * mask).sum()
            observed_count = mask.sum()
            loss = error_sum / observed_count.clamp_min(1)
            loss.backward()
            optimizer.step()

            epoch_error += error_sum.detach()
            epoch_observed += observed_count

        train_rmse = (epoch_error / epoch_observed).sqrt().item()
        message = f"Epoch {epoch:02d}/{epoch_count} | Train log RMSE {train_rmse:.4f}"
        if validation_origin is not None:
            score = forecast_rmsle(predict(model, validation_origin, context), validation_origin)
            message += f" | Validation RMSLE {score:.4f}"
            if score < best_score:
                best_score, best_epoch = score, epoch
                best_weights = {
                    name: value.detach().cpu().clone()
                    for name, value in model.state_dict().items()
                }
        if verbose:
            print(message, flush=True)

    if validation_origin is not None:
        if best_weights is None:
            raise ValueError("Training did not produce a finite validation RMSLE.")
        model.load_state_dict(best_weights)
    return model, best_epoch, best_score

In [78]:
train_origins = training_origins(split_day)
assert train_origins[-1] + horizon <= split_day
validation_context, _ = fit_context(train_origins, raw_context)
baseline_model = SalesForecaster("last_year" in feature_groups).to(device)
baseline_rmsle = forecast_rmsle(predict(baseline_model, split_day, validation_context), split_day)
input_width = baseline_model.input_width
del baseline_model

print(f"Training examples: {len(train_origins) * series_count:,} | MLP input width: {input_width}")
print(f"Same-weekday baseline RMSLE: {baseline_rmsle:.4f}", flush=True)
model, best_epoch, best_rmsle = fit_model(
    train_origins, validation_context, epochs, validation_origin=split_day,
)
print(f"Selected epoch {best_epoch} | Validation RMSLE {best_rmsle:.4f}", flush=True)

Training examples: 572,022 | MLP input width: 146
Same-weekday baseline RMSLE: 0.5398
Epoch 01/8 | Train log RMSE 0.6149 | Validation RMSLE 0.4524
Epoch 02/8 | Train log RMSE 0.5639 | Validation RMSLE 0.4478
Epoch 03/8 | Train log RMSE 0.5424 | Validation RMSLE 0.4470
Epoch 04/8 | Train log RMSE 0.5302 | Validation RMSLE 0.4438
Epoch 05/8 | Train log RMSE 0.5205 | Validation RMSLE 0.4311
Epoch 06/8 | Train log RMSE 0.5127 | Validation RMSLE 0.4332
Epoch 07/8 | Train log RMSE 0.5061 | Validation RMSLE 0.4313
Epoch 08/8 | Train log RMSE 0.5001 | Validation RMSLE 0.4214
Selected epoch 8 | Validation RMSLE 0.4214


In [79]:
def compare_transaction_windows(window_sizes=(14, 28, 56), seeds=(42, 43, 44),
                                *, include_last_year=False):
    if feature_groups != ("transactions",):
        raise ValueError('Set feature_groups = ("transactions",) for a controlled comparison.')

    cutoff_days = (split_day - horizon, split_day)
    rows = []
    for window in window_sizes:
        features = make_raw_context(window)
        for cutoff in cutoff_days:
            origins = training_origins(cutoff)
            context, _ = fit_context(origins, features)
            for trial_seed in seeds:
                candidate, epoch, rmsle = fit_model(
                    origins, context, epochs, validation_origin=cutoff,
                    model_seed=trial_seed, include_last_year=include_last_year, verbose=False,
                )
                del candidate
                validation_date = calendar["date"][cutoff]
                rows.append({
                    "transaction_window": window,
                    "validation_start": validation_date,
                    "seed": trial_seed,
                    "best_epoch": epoch,
                    "rmsle": rmsle,
                })
                print(f"{window:2d} days | {validation_date} | seed {trial_seed} "
                      f"| epoch {epoch} | RMSLE {rmsle:.4f}", flush=True)
    return pl.DataFrame(rows)


if run_window_comparison:
    window_scores = compare_transaction_windows()
    print(window_scores.group_by("transaction_window").agg(
        pl.col("rmsle").mean().alias("mean_rmsle")
    ).sort("mean_rmsle"))

In [80]:
full_origins = training_origins(test_day, include_last=True)
assert full_origins[-1] + horizon == test_day
final_context, context_scaler = fit_context(full_origins, raw_context)
final_model, _, _ = fit_model(full_origins, final_context, best_epoch)

sales_predictions = torch.expm1(predict(final_model, test_day, final_context)).cpu().numpy()
assert sales_predictions.shape == (series_count, horizon)
assert np.isfinite(sales_predictions).all()

test_ids = unified.filter(pl.col("split") == "test").sort(series_day).select("id")
forecasts = test_ids.with_columns(pl.Series("sales", sales_predictions.ravel()))
submission = sample.select("id").join(
    forecasts, on="id", how="left", validate="1:1", maintain_order="left"
)
assert submission.height == test.height
assert submission["sales"].null_count() == 0

output_dir.mkdir(parents=True, exist_ok=True)
submission.write_csv(output_dir / "submission.csv")
torch.save({
    "model_state_dict": {name: value.cpu() for name, value in final_model.state_dict().items()},
    "series_keys": series_keys.rows(),
    "lookback": lookback,
    "horizon": horizon,
    "hidden_widths": hidden_widths,
    "input_width": final_model.input_width,
    "feature_groups": feature_groups,
    "transaction_window": transaction_window if "transactions" in feature_groups else None,
    "annual_lag": annual_lag if final_model.include_last_year else None,
    "calendar_features": calendar_names,
    "context_features": context_names,
    "context_scaler": context_scaler,
    "store_features": store_feature_names,
    "seed": seed,
    "epochs": best_epoch,
    "validation_rmsle": best_rmsle,
}, output_dir / "model.pt")
print(f"Refit through {last_labeled_date}. Saved {submission.height:,} forecasts to {output_dir}.")
submission.head()

Epoch 01/8 | Train log RMSE 0.6134
Epoch 02/8 | Train log RMSE 0.5625
Epoch 03/8 | Train log RMSE 0.5415
Epoch 04/8 | Train log RMSE 0.5291
Epoch 05/8 | Train log RMSE 0.5192
Epoch 06/8 | Train log RMSE 0.5113
Epoch 07/8 | Train log RMSE 0.5044
Epoch 08/8 | Train log RMSE 0.4984
Refit through 2017-08-15. Saved 28,512 forecasts to d:\Coding\yumeko\data\output\transactions.


id,sales
i64,f32
3000888,3.891994
3000889,0.042973
3000890,4.456435
3000891,2515.469482
3000892,0.045149
